# Joinery: baseline evaluation on Spider dev

Thin wrapper around `scripts/run_eval.py`. All logic lives in the `joinery` package.

**Before running**
1. Settings: Accelerator = GPU T4 (or P100), Internet = on.
2. Add the Spider dataset as an input (see the README, "Data setup"), then set `SPIDER_DIR` below.
3. To resume an interrupted run, add the previous notebook version's output as an input and set
   `RESUME_FROM` to its `results` folder. Completed examples are skipped.

In [ ]:
CONFIG = "eval_qwen15b_instruct"  # or eval_qwen15b_coder, eval_qwen7b_coder_4bit
SPIDER_DIR = "/kaggle/input/spider-data/spider_data"  # adjust to your dataset's path
RESUME_FROM = None  # e.g. "/kaggle/input/<previous-notebook-output>/results"
EXTRA_OVERRIDES = []  # e.g. ["data.limit=20", "run_name=smoke"]
REPO_URL = "https://github.com/yukta1103/joinery.git"
OUTPUT_DIR = "/kaggle/working/results"

In [ ]:
!git clone --depth 1 {REPO_URL} /kaggle/working/joinery
# --no-deps keeps Kaggle's preinstalled torch/transformers; only add what is missing.
!pip install -q sqlglot
!pip install -q --no-deps -e /kaggle/working/joinery
if CONFIG.endswith("4bit"):
    !pip install -q bitsandbytes

In [ ]:
import shutil
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, OUTPUT_DIR, dirs_exist_ok=True)

In [ ]:
overrides = [f"data.spider_dir={SPIDER_DIR}", f"output_dir={OUTPUT_DIR}", *EXTRA_OVERRIDES]
args = " ".join(f"--set '{o}'" for o in overrides)
!cd /kaggle/working/joinery && python scripts/run_eval.py --config configs/{CONFIG}.yaml {args}